In [ ]:
import re
from pathlib import Path

import pandas as pd
import pymupdf

# find the repo root whether the notebook is run from the repo root or from members/team_huy/
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "tmp" / "RAG-data").exists())
PDF_DIR = ROOT / "tmp" / "RAG-data" / "Research Paper"
CHARS_PER_TOKEN = 4  # rough estimate for English text (no tokenizer installed)
ROLES = ["abstract", "introduction", "conclusion", "references"]

NUMBERED = re.compile(r"^(?:\d{1,2}|[A-Z])\.?\s+[A-Z][^.!?]{2,70}$")  # "3 Model Architecture", "A Details"
UNNUMBERED = re.compile(
    r"^(?:abstract|introduction|acknowledg(?:e)?ments?(?:\s+and\s+\w+)?|references|bibliography|appendix(?:\s+[A-Z](?::.*)?)?"
    r"|supplementary.*|conclusions?"
    r"|(?:discussion|limitations?)(?:\s+(?:and|&)\s+[a-z ]{3,30})?|future (?:work|directions)"
    r"|ethic(?:s|al)(?:\s+(?:and\s+\w+\s+)?(?:statement|considerations?))?|broader impacts?|impact statement"
    r"|reproducibility statement|author contributions?|contributions?|(?:core )?contributors(?:\s+and\s+\w+)?|funding)$",
    re.I,
)
BARE_NUM = re.compile(r"^(?:\d{1,2}|[A-Z])\.?$")  # "1", "3.", "A": number alone on its own line
NOT_HEADING = re.compile(r"^(?:table|figure|fig\.|algorithm|listing|example)\b", re.I)


def role_of(heading):
    """Map a heading to abstract / introduction / conclusion / references / ... / body."""
    h = re.sub(r"^(?:\d{1,2}|[A-Z])\.?\s+", "", heading).strip().lower()
    if h.startswith("abstract"):
        return "abstract"
    if "introduction" in h:
        return "introduction"
    if h.startswith(("reference", "bibliography")):
        return "references"
    if h.startswith("acknowledg"):
        return "acknowledgements"
    if "conclusion" in h or h.startswith("concluding"):
        return "conclusion"
    if h.startswith(("appendix", "supplementary")):
        return "appendix"
    if h.startswith(("discussion", "limitation", "future work", "future direction")):
        return "discussion"
    if h.startswith(("ethic", "broader impact", "impact statement", "reproducibility", "author contribution",
                     "contribution", "core contributors", "contributors", "funding")):
        return "back_matter"  # only here so that it ends the section before it
    return "body"


def keep_in_sequence(heads):
    """Drop numbered 'headings' that are really chart labels, affiliations or list items.

    Top-level numbers must increase (1, 2, 3 ...); appendix letters (A, B, ...) are only
    accepted after References/Appendix and must also increase. Unnumbered headings always stay.
    """
    kept, last_num, last_letter, in_back = [], 0, 0, False
    # numbering starts at the numbered Introduction; earlier "1 University of ..." lines are affiliations
    first_intro = next((i for i, h in enumerate(heads)
                        if re.match(r"^\d{1,2}\.?\s+introduction", h["heading"], re.I)), 0)
    for idx, h in enumerate(heads):
        if idx < first_intro and re.match(r"^\d", h["heading"]):
            continue
        if h["heading"].endswith((",", ":")) or re.search(
                r"^\S+\s+(?:thousand|million|billion|trillion)\b", h["heading"], re.I):
            continue
        m = re.match(r"^(\d{1,2}|[A-Z])\.?\s", h["heading"])
        role = role_of(h["heading"])
        if role in ("conclusion", "discussion"):
            pass  # "5. Discussion & Conclusion" is never a chart label, even if "5" was already used
        elif role in ("references", "appendix"):
            in_back = True  # "9 References" etc. are accepted whatever their number
        elif m and m.group(1).isdigit():
            n = int(m.group(1))
            if not last_num < n <= last_num + 2:
                continue
            last_num = n
        elif m:
            letter = ord(m.group(1)) - 64
            if not in_back or not last_letter < letter <= last_letter + 2:
                continue
            last_letter = letter
        if kept and kept[-1]["heading"] == h["heading"]:
            continue
        kept.append(h)
    return kept


def find_headings(pages):
    """Headings as dicts: heading, offset into the full text.

    PyMuPDF often emits the section number and the title as two lines ("1", "Introduction"),
    so a bare number line is glued to the next non-empty line before matching.
    """
    heads, tocs, offset = [], [], 0
    for text in pages:
        lines = text.split("\n")
        consumed = -1  # index of a title line already merged into the previous number line
        for i, line in enumerate(lines):
            s = line.strip()
            start = offset
            offset += len(line) + 1
            if i == consumed:
                continue
            if BARE_NUM.match(s):
                j = next((k for k in range(i + 1, min(i + 3, len(lines))) if lines[k].strip()), None)
                nxt = lines[j].strip() if j is not None else ""
                if not nxt or NOT_HEADING.match(nxt) or len(nxt) > 70 or len(nxt.split()) > 8:
                    continue
                s = f"{s.rstrip('.')} {nxt}"
                consumed = j
            if 3 <= len(s) <= 80 and not NOT_HEADING.match(s) and (UNNUMBERED.match(s) or NUMBERED.match(s)):
                after = next((l.strip() for l in lines[max(i, consumed) + 1:max(i, consumed) + 3] if l.strip()), "")
                if re.fullmatch(r"\d{1,3}", after):
                    # table-of-contents entry (title followed by its page number): it still ends the
                    # section before it, but is kept out of the numbering logic and never extracted
                    if role_of(s) != "body":  # chart labels like "9 Device 1" must not cut sections
                        tocs.append({"heading": s, "offset": start, "toc": True})
                else:
                    heads.append({"heading": s, "offset": start})
    return sorted(keep_in_sequence(heads) + tocs, key=lambda h: h["offset"])


def extract_sections(pages, roles=ROLES):
    """Return {role: text} for the wanted roles ('' if the paper has no such heading).

    A role can span several detected sections, so their texts are joined. When a paper has no
    "Abstract" heading (e.g. Chinchilla, Llama 3), the front matter is used as the abstract.
    When it has no "Conclusion" heading, its closing "Discussion" / "Limitations" / "Future Work"
    section is used as the conclusion.
    """
    text = "".join(line + "\n" for p in pages for line in p.split("\n"))  # offsets refer to this text
    heads = find_headings(pages)
    # only the last "References" heading counts (earlier ones are a table of contents or mentions)
    ref_idx = [i for i, h in enumerate(heads) if role_of(h["heading"]) == "references" and not h.get("toc")]
    heads = [h for i, h in enumerate(heads) if i not in ref_idx[:-1]]

    bounds = [{"heading": "(front matter)", "offset": 0}] + heads
    parts = {}  # role -> list of section texts
    for i, h in enumerate(bounds):
        end = bounds[i + 1]["offset"] if i + 1 < len(bounds) else len(text)
        role = "front_matter" if i == 0 else "toc" if h.get("toc") else role_of(h["heading"])
        if role not in ("references", "toc") and any(
                role_of(b["heading"]) == "references" and not b.get("toc") and b["offset"] < h["offset"] for b in heads):
            role = "appendix"
        parts.setdefault(role, []).append(text[h["offset"]:end].strip())

    out = {r: "\n".join(parts.get(r, [])) for r in roles}
    if "conclusion" in out and not out["conclusion"]:
        out["conclusion"] = "\n".join(parts.get("discussion", [])[-1:])  # last one: earlier hits are the table of contents
    if "abstract" in out and not out["abstract"]:
        out["abstract"] = "\n".join(parts["front_matter"])
    return out


In [ ]:
stats, texts = [], []
for pdf in sorted(PDF_DIR.glob("*.pdf")):
    with pymupdf.open(pdf) as doc:
        pages = [page.get_text() for page in doc]
    sections = extract_sections(pages)

    stats.append({
        "file": pdf.name,
        "file_size_mb": round(pdf.stat().st_size / 1e6, 2),
        "pages": len(pages),
        "est_tokens": len("\n".join(pages)) // CHARS_PER_TOKEN,
        **{f"{role}_tokens": len(text) // CHARS_PER_TOKEN for role, text in sections.items()},
    })
    texts.append({"file": pdf.name, **sections})

df = pd.DataFrame(stats)        # one combined table: file size, tokens, tokens per section
df_text = pd.DataFrame(texts)   # the extracted section text itself

print(f"{len(df)} files, {df.file_size_mb.sum():,.1f} MB, {df.est_tokens.sum():,} estimated tokens in total")
df
